In [1]:
import pandas as pd

df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

print(df.shape)
print(df.dtypes)

(7043, 21)
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object


In [2]:
# 1. Les valeurs vides cachées dans TotalCharges
vides = df[df["TotalCharges"].str.strip() == ""]
print("Lignes avec TotalCharges vide :", len(vides))
print(vides[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]])

# 2. Valeurs manquantes et doublons
print("\nValeurs manquantes par colonne :")
print(df.isna().sum().sum(), "au total")
print("Doublons :", df.duplicated().sum())
print("customerID en double :", df["customerID"].duplicated().sum())

# 3. Répartition de la cible
print("\nRépartition de Churn :")
print(df["Churn"].value_counts())
print(df["Churn"].value_counts(normalize=True).round(3))

Lignes avec TotalCharges vide : 11
      customerID  tenure  MonthlyCharges TotalCharges Churn
488   4472-LVYGI       0           52.55                 No
753   3115-CZMZD       0           20.25                 No
936   5709-LVOEQ       0           80.85                 No
1082  4367-NUYAO       0           25.75                 No
1340  1371-DWPAZ       0           56.05                 No
3331  7644-OMVMY       0           19.85                 No
3826  3213-VVOLG       0           25.35                 No
4380  2520-SGTTA       0           20.00                 No
5218  2923-ARZLG       0           19.70                 No
6670  4075-WKNIU       0           73.35                 No
6754  2775-SEFEE       0           61.90                 No

Valeurs manquantes par colonne :
0 au total
Doublons : 0
customerID en double : 0

Répartition de Churn :
Churn
No     5174
Yes    1869
Name: count, dtype: int64
Churn
No     0.735
Yes    0.265
Name: proportion, dtype: float64


In [3]:
# 1. Convertir TotalCharges en nombre (les vides deviennent 0)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].str.strip(), errors="coerce").fillna(0)

# 2. Churn en 0/1
df["ChurnFlag"] = (df["Churn"] == "Yes").astype(int)

# 3. Vérifications
print(df["TotalCharges"].dtype)
print("Valeurs manquantes :", df.isna().sum().sum())
print("Taux de churn :", round(df["ChurnFlag"].mean() * 100, 1), "%")

# 4. Sauvegarder le fichier nettoyé
df.to_csv("../data/telco_clean.csv", index=False)
print("Fichier sauvegardé :", df.shape)

float64
Valeurs manquantes : 0
Taux de churn : 26.5 %
Fichier sauvegardé : (7043, 22)


In [4]:
import sqlite3

# Charger le CSV propre dans une base SQLite
conn = sqlite3.connect("../data/telco.db")
df.to_sql("customers", conn, if_exists="replace", index=False)

def sql(requete):
    return pd.read_sql_query(requete, conn)

# Question 1 : quel type de contrat perd le plus de clients ?
sql("""
SELECT Contract,
       COUNT(*) AS nb_clients,
       SUM(ChurnFlag) AS nb_partis,
       ROUND(100.0 * SUM(ChurnFlag) / COUNT(*), 1) AS taux_churn_pct
FROM customers
GROUP BY Contract
ORDER BY taux_churn_pct DESC
""")

,Contract,nb_clients,nb_partis,taux_churn_pct
0,Month-to-month,3875,1655,42.7
1,One year,1473,166,11.3
2,Two year,1695,48,2.8


In [5]:
sql("""
SELECT CASE
         WHEN tenure <= 12 THEN '1. 0-12 mois'
         WHEN tenure <= 24 THEN '2. 13-24 mois'
         WHEN tenure <= 48 THEN '3. 25-48 mois'
         ELSE '4. plus de 48 mois'
       END AS anciennete,
       COUNT(*) AS nb_clients,
       SUM(ChurnFlag) AS nb_partis,
       ROUND(100.0 * SUM(ChurnFlag) / COUNT(*), 1) AS taux_churn_pct
FROM customers
GROUP BY anciennete
ORDER BY anciennete
""")

,anciennete,nb_clients,nb_partis,taux_churn_pct
0,1. 0-12 mois,2186,1037,47.4
1,2. 13-24 mois,1024,294,28.7
2,3. 25-48 mois,1594,325,20.4
3,4. plus de 48 mois,2239,213,9.5


In [6]:
sql("""
SELECT InternetService,
       TechSupport,
       COUNT(*) AS nb_clients,
       ROUND(100.0 * SUM(ChurnFlag) / COUNT(*), 1) AS taux_churn_pct
FROM customers
WHERE InternetService <> 'No'
GROUP BY InternetService, TechSupport
ORDER BY taux_churn_pct DESC
""")

,InternetService,TechSupport,nb_clients,taux_churn_pct
0,Fiber optic,No,2230,49.4
1,DSL,No,1243,27.8
2,Fiber optic,Yes,866,22.6
3,DSL,Yes,1178,9.7


In [7]:
from scipy.stats import chi2_contingency

def test_khi2(colonne):
    tableau = pd.crosstab(df[colonne], df["Churn"])
    chi2, p, ddl, _ = chi2_contingency(tableau)
    print(f"{colonne:18s} khi2 = {chi2:8.1f} | p-value = {p:.2e}")

for col in ["Contract", "InternetService", "TechSupport", "PaymentMethod", "gender"]:
    test_khi2(col)

Contract           khi2 =   1184.6 | p-value = 5.86e-258
InternetService    khi2 =    732.3 | p-value = 9.57e-160
TechSupport        khi2 =    828.2 | p-value = 1.44e-180
PaymentMethod      khi2 =    648.1 | p-value = 3.68e-140
gender             khi2 =      0.5 | p-value = 4.87e-01


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

X = df.drop(columns=["customerID", "Churn", "ChurnFlag"])
y = df["ChurnFlag"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

num = ["tenure", "MonthlyCharges", "TotalCharges"]
cat = [c for c in X.columns if c not in num]

prep = ColumnTransformer([
    ("num", StandardScaler(), num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
])

modele = Pipeline([
    ("prep", prep),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])

modele.fit(X_train, y_train)
pred = modele.predict(X_test)
proba = modele.predict_proba(X_test)[:, 1]

print(classification_report(y_test, pred, target_names=["Reste", "Part"]))
print("ROC-AUC :", round(roc_auc_score(y_test, proba), 3))

              precision    recall  f1-score   support

       Reste       0.90      0.72      0.80      1035
        Part       0.50      0.78      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409

ROC-AUC : 0.842


In [9]:
from sklearn.ensemble import RandomForestClassifier

rf = Pipeline([
    ("prep", prep),
    ("clf", RandomForestClassifier(
        n_estimators=300, min_samples_leaf=5,
        class_weight="balanced", random_state=42, n_jobs=-1)),
])

rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
proba_rf = rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, pred_rf, target_names=["Reste", "Part"]))
print("ROC-AUC :", round(roc_auc_score(y_test, proba_rf), 3))

              precision    recall  f1-score   support

       Reste       0.90      0.76      0.82      1035
        Part       0.54      0.77      0.63       374

    accuracy                           0.76      1409
   macro avg       0.72      0.77      0.73      1409
weighted avg       0.81      0.76      0.77      1409

ROC-AUC : 0.842


In [10]:
noms = modele.named_steps["prep"].get_feature_names_out()
coefs = modele.named_steps["clf"].coef_[0]
imp = pd.Series(coefs, index=noms).sort_values()

print("Facteurs qui AUGMENTENT le risque de départ :")
print(imp.tail(8).round(2))
print("\nFacteurs qui DIMINUENT le risque de départ :")
print(imp.head(8).round(2))

Facteurs qui AUGMENTENT le risque de départ :
cat__TechSupport_No                    0.17
cat__OnlineSecurity_No                 0.21
cat__PaymentMethod_Electronic check    0.24
cat__StreamingTV_Yes                   0.27
cat__StreamingMovies_Yes               0.28
num__TotalCharges                      0.47
cat__Contract_Month-to-month           0.66
cat__InternetService_Fiber optic       0.72
dtype: float64

Facteurs qui DIMINUENT le risque de départ :
num__tenure                                 -1.14
cat__Contract_Two year                      -0.77
num__MonthlyCharges                         -0.68
cat__InternetService_DSL                    -0.62
cat__OnlineSecurity_No internet service     -0.27
cat__OnlineBackup_No internet service       -0.27
cat__InternetService_No                     -0.27
cat__DeviceProtection_No internet service   -0.27
dtype: float64


In [11]:
df_pbi = df.copy()
df_pbi["risque_depart"] = modele.predict_proba(X)[:, 1].round(3)
df_pbi["tranche_anciennete"] = pd.cut(
    df_pbi["tenure"], bins=[-1, 12, 24, 48, 100],
    labels=["0-12 mois", "13-24 mois", "25-48 mois", "plus de 48 mois"])
df_pbi["niveau_risque"] = pd.cut(
    df_pbi["risque_depart"], bins=[0, 0.3, 0.6, 1],
    labels=["Faible", "Moyen", "Eleve"], include_lowest=True)

df_pbi.to_csv("../data/telco_powerbi.csv", index=False)
print(df_pbi["niveau_risque"].value_counts())

niveau_risque
Faible    2977
Eleve     2348
Moyen     1718
Name: count, dtype: int64
